# MCP (Model Context Protocol) 간단 실습

이 노트북은 **MCP(Model Context Protocol)** 의 기본 개념을 직접 코드로 체험합니다.

지난 시간에는 `@tool` 로 도구를 **같은 파이썬 프로세스 안에** 정의했습니다.
MCP는 한 걸음 더 나아가, 도구를 **별도의 "서버"** 로 분리하고
LLM 앱("클라이언트")이 **표준 프로토콜**로 그 서버에 연결해 도구를 가져다 씁니다.

```
[LLM 앱 = MCP 클라이언트]  <--- 표준 프로토콜(JSON-RPC) --->  [MCP 서버 = 도구 모음]
```

이렇게 분리하면 같은 도구 서버를 Claude Desktop, Cursor, LangChain 등 **어떤 클라이언트에서도 재사용**할 수 있습니다.

---
**API 키 없이 무료로 동작하는 실습입니다**

| 장 | 내용 | OpenAI 키 |
|----|------|:--------:|
| 1 | 패키지 설치 / 환경 | 불필요 |
| 2 | FastMCP 로 MCP 서버 만들기 | 불필요 |
| 3 | 순수 MCP 클라이언트로 도구 직접 호출 | 불필요 |
| 4 | MCP 도구를 LangChain 도구로 변환 | 불필요 |

> MCP의 핵심(서버-클라이언트, 도구 목록 조회/호출)을 전부 체험할 수 있습니다.

## 1. 패키지 설치 및 환경 설정

추가로 필요한 패키지는 두 개뿐이며, 둘 다 **무료 오픈소스**입니다.

- `mcp` — MCP 공식 Python SDK (서버/클라이언트 모두 포함, FastMCP 내장)
- `langchain-mcp-adapters` — MCP 서버의 도구를 LangChain 도구로 바꿔주는 어댑터

In [ ]:
%pip install -q -r ../requirements.txt
%pip install -q mcp langchain-mcp-adapters

import sys
print("Python:", sys.version.split()[0])   # MCP는 Python 3.10+ 필요

# MCP 자체 실습(2~4장)은 API 키가 전혀 필요 없습니다.


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Python: 3.12.1
OPENAI_API_KEY: 설정됨 (5장 실행 가능)


## 2. MCP 서버 만들기 (FastMCP)

`FastMCP` 를 쓰면 `@mcp.tool()` 데코레이터만으로 MCP 서버를 만들 수 있습니다.
4번 노트북의 `@tool` 과 거의 똑같이 생겼지만, 이 함수들은 **별도 서버 프로세스**에서 실행됩니다.

여기서 만드는 도구는 **외부 네트워크/API 키가 전혀 필요 없는 도구**입니다.
- `add` — 두 숫자 더하기
- `calculate` — 수식 계산
- `now` — 현재 시각
- `text_stats` — 문자열 통계

> MCP 서버는 보통 별도 파일로 두고 프로그램처럼 실행합니다.
> 그래서 아래 셀은 `%%writefile` 매직으로 서버 코드를 `mcp_demo_server.py` 파일로 저장합니다.

In [7]:
%%writefile mcp_demo_server.py
"""아주 간단한 MCP 데모 서버.
외부 네트워크나 API 키가 전혀 필요 없는 도구만 제공합니다.
직접 실행: python mcp_demo_server.py 
"""
import math
from datetime import datetime

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo-tools")


@mcp.tool()
def add(a: float, b: float) -> float:
    """두 숫자를 더합니다."""
    return a + b


@mcp.tool()
def calculate(expression: str) -> str:
    """수학 표현식을 계산합니다. sqrt, pow, pi, log 등을 지원합니다.
    예: '2 + 3 * 4', 'sqrt(16)', 'pi * 5**2'"""
    safe = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
    try:
        return str(eval(expression, {"__builtins__": {}}, safe))
    except Exception as e:
        return f"계산 오류: {e}"


@mcp.tool()
def now() -> str:
    """서버 기준 현재 날짜와 시각을 ISO 형식 문자열로 반환합니다."""
    return datetime.now().isoformat(timespec="seconds")


@mcp.tool()
def text_stats(text: str) -> dict:
    """문자열의 글자 수, 공백 제외 글자 수, 단어 수를 반환합니다."""
    return {
        "characters": len(text),
        "characters_no_space": len(text.replace(" ", "")),
        "words": len(text.split()),
    }


# (참고) MCP는 도구(tool) 외에 '리소스(resource)'도 노출할 수 있습니다.
@mcp.resource("info://server")
def server_info() -> str:
    """이 데모 서버를 설명하는 리소스."""
    return "MCP 데모 서버: add / calculate / now / text_stats 도구를 제공합니다."


if __name__ == "__main__":
    mcp.run()  # transport 미지정 시 stdio 사용

Overwriting mcp_demo_server.py


## 3. MCP 클라이언트로 도구 직접 호출

이제 **MCP 공식 SDK의 클라이언트**로 위 서버에 연결합니다. LLM은 전혀 사용하지 않습니다.

흐름은 항상 동일합니다.
1. `stdio_client(...)` 로 서버를 프로세스로 띄우고 통신 채널을 연다
2. `ClientSession` 을 만들고 `initialize()`
3. `list_tools()` 로 서버가 제공하는 도구 목록을 받아온다
4. `call_tool(이름, 인자)` 로 도구를 실행한다


In [8]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command=sys.executable,         # 현재 커널과 같은 파이썬으로 서버 실행
    args=["mcp_demo_server.py"],    # 2장에서 만든 서버 파일
)

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        # (1) 서버가 노출한 도구 목록
        tool_list = await session.list_tools()
        print("=== MCP 서버가 제공하는 도구 ===")
        for t in tool_list.tools:
            print(f"- {t.name}: {t.description}")

        # (2) 도구 직접 호출 (반환값은 result.content[0].text 에 들어 있음)
        print("\n=== 도구 호출 결과 ===")
        r = await session.call_tool("add", {"a": 3, "b": 5})
        print("add(3, 5)               ->", r.content[0].text)

        r = await session.call_tool("calculate", {"expression": "pi * 7**2"})
        print("calculate('pi * 7**2')  ->", r.content[0].text)

        r = await session.call_tool("now", {})
        print("now()                   ->", r.content[0].text)

        r = await session.call_tool("text_stats", {"text": "MCP 실습 재미있다"})
        print("text_stats(...)         ->", r.content[0].text)

        # (3) 리소스도 읽어볼 수 있습니다 (도구가 아닌 또 다른 MCP 기능)
        try:
            res = await session.read_resource("info://server")
            print("\nresource info://server  ->", res.contents[0].text)
        except Exception as e:
            print("\n(리소스 읽기 생략:", e, ")")

=== MCP 서버가 제공하는 도구 ===
- add: 두 숫자를 더합니다.
- calculate: 수학 표현식을 계산합니다. sqrt, pow, pi, log 등을 지원합니다.
    예: '2 + 3 * 4', 'sqrt(16)', 'pi * 5**2'
- now: 서버 기준 현재 날짜와 시각을 ISO 형식 문자열로 반환합니다.
- text_stats: 문자열의 글자 수, 공백 제외 글자 수, 단어 수를 반환합니다.

=== 도구 호출 결과 ===
add(3, 5)               -> 8.0
calculate('pi * 7**2')  -> 153.93804002589985
now()                   -> 2026-05-29T05:10:55
text_stats(...)         -> {
  "characters": 11,
  "characters_no_space": 9,
  "words": 3
}

resource info://server  -> MCP 데모 서버: add / calculate / now / text_stats 도구를 제공합니다.


## 4. MCP 도구를 LangChain 도구로 변환

`langchain-mcp-adapters` 의 `MultiServerMCPClient` 를 쓰면
MCP 서버의 도구들을 **그대로 LangChain Tool 객체로** 가져올 수 있습니다.
(이 단계까지도 OpenAI 키는 필요 없습니다 — 도구를 불러오기만 함)

여러 서버를 동시에 등록할 수 있어서 이름이 `MultiServer...` 입니다.

In [9]:
import sys
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "demo-tools": {
            "command": sys.executable,
            "args": ["mcp_demo_server.py"],
            "transport": "stdio",
        }
        # 필요하면 여기에 다른 MCP 서버를 추가로 등록할 수 있습니다.
    }
)

# MCP 서버의 도구들을 LangChain Tool 로 변환
mcp_tools = await mcp_client.get_tools()
print(f"불러온 MCP 도구 {len(mcp_tools)}개:")
for t in mcp_tools:
    print(f"- {t.name}: {t.description}")


def to_text(result):
    """MCP 도구 결과를 읽기 좋은 문자열로 정규화합니다.
    라이브러리 버전에 따라 결과가 문자열일 수도, 콘텐츠 블록 리스트
    ([{'type': 'text', 'text': ...}]) 일 수도 있어 양쪽을 모두 처리합니다."""
    if isinstance(result, list):
        out = []
        for block in result:
            if isinstance(block, dict):
                out.append(block.get("text", str(block)))
            else:
                out.append(getattr(block, "text", str(block)))
        return "\n".join(out)
    return str(result)


# LangChain 도구처럼 단독 호출도 가능 (MCP 도구는 비동기 -> ainvoke 사용)
tools_by_name = {t.name: t for t in mcp_tools}
print("\n[단독 호출] add(10, 32) ->", to_text(await tools_by_name["add"].ainvoke({"a": 10, "b": 32})))

불러온 MCP 도구 4개:
- add: 두 숫자를 더합니다.
- calculate: 수학 표현식을 계산합니다. sqrt, pow, pi, log 등을 지원합니다.
    예: '2 + 3 * 4', 'sqrt(16)', 'pi * 5**2'
- now: 서버 기준 현재 날짜와 시각을 ISO 형식 문자열로 반환합니다.
- text_stats: 문자열의 글자 수, 공백 제외 글자 수, 단어 수를 반환합니다.

[단독 호출] add(10, 32) -> 42.0


## 정리

| 개념 | 설명 |
|------|------|
| **MCP** | LLM 앱이 외부 도구/데이터에 연결하는 **표준 프로토콜** (JSON-RPC 기반) |
| **MCP 서버** | 도구·리소스·프롬프트를 노출하는 쪽. `FastMCP` + `@mcp.tool()` 로 간단히 작성 |
| **MCP 클라이언트** | 서버에 연결해 도구를 가져다 쓰는 LLM 앱 쪽 (`ClientSession`) |
| **트랜스포트** | 통신 방식. 로컬은 `stdio`(자식 프로세스), 원격은 `streamable-http` |
| `list_tools()` / `call_tool()` | 도구 목록 조회 / 도구 실행 (프로토콜의 핵심 동작) |
| `MultiServerMCPClient` | MCP 도구를 LangChain Tool 로 변환해 기존 Agent에 그대로 연결 |

**4번 노트북의 `@tool` 과의 차이**

| | 4번 노트북 (`@tool`) | 6번 노트북 (MCP) |
|---|---|---|
| 도구 위치 | 같은 프로세스 안 | **별도 서버 프로세스** |
| 재사용성 | 이 앱에서만 | **Claude Desktop, Cursor 등 어떤 MCP 클라이언트에서도** |
| 호출 방식 | `tool.invoke` | 프로토콜(`call_tool`) → 어댑터로 `ainvoke` |


> 핵심: 한 번 만든 MCP 서버는 **어떤 LLM/클라이언트에서도** 재사용된다는 점입니다.